# PM2.5 filtering of smoke-classified days

Move every SR/CI-filtered observation from a smoke-classified calendar day into the no-smoke dataset only when **all available hourly mean PM2.5 values on that day are below 9 µg/m³**. If any hourly mean is 9 µg/m³ or higher, the full day remains smoke-classified. Hourly PM2.5 follows the repository convention: AQS parameter 88101, 1-hour samples, POCs 2 and 4, averaged when both instruments are available.

## Context & Methods

### Key assumptions
- A day is defined by the local calendar date in `dt`.
- Only valid numeric POC 2/4 readings contribute to an hourly mean; pandas averages the available reading if one instrument is missing.
- Missing PM2.5 hours do not trigger movement. Every smoke day must have at least one valid hourly PM2.5 value.
- The original files are never overwritten.

In [15]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'Notebooks' else Path.cwd().resolve()
DATA_ROOT = ROOT / 'data/utah/data_up_to_2025'
SMOKE_PATH = DATA_ROOT / 'new_canister_breakout_2026_06_23/Filtered_Canister_data/Smoke/canister_voc_matched_windows_all_smoke.csv'
NO_SMOKE_PATH = DATA_ROOT / 'new_canister_breakout_2026_06_23/Filtered_Canister_data/No_smoke/canister_voc_matched_windows_all_no_smoke.csv'
RAW_PATH = DATA_ROOT / 'raw/HW_data_raw_2026-05-23.csv'
OUTPUT_DIR = DATA_ROOT / 'new_canister_breakout_2026_06_23/Filtered_Canister_data/PM2.5_Smoke'
SMOKE_OUTPUT = OUTPUT_DIR / '2025_summed_smoke_all_PM2.5_filtered.csv'
NO_SMOKE_OUTPUT = OUTPUT_DIR / '2025_summed_no_smoke_all_PM2.5_filtered.csv'
THRESHOLD = 9.0
POCS = [2, 4]

## Data

In [16]:
smoke = pd.read_csv(SMOKE_PATH)
no_smoke = pd.read_csv(NO_SMOKE_PATH)
assert list(smoke.columns) == list(no_smoke.columns), 'Input schemas differ.'
assert not smoke['dt'].duplicated().any(), 'Smoke input contains duplicate timestamps.'
assert not no_smoke['dt'].duplicated().any(), 'No-smoke input contains duplicate timestamps.'
assert set(smoke['dt']).isdisjoint(no_smoke['dt']), 'Inputs already overlap by timestamp.'

smoke['_date'] = smoke['dt'].astype(str).str[:10]
smoke_dates = set(smoke['_date'])
print(f'Smoke input: {len(smoke):,} rows across {len(smoke_dates):,} days')
print(f'No-smoke input: {len(no_smoke):,} rows')

Smoke input: 48 rows across 39 days
No-smoke input: 86 rows


In [17]:
columns = ['parameter_code', 'poc', 'date_local', 'time_local', 'sample_measurement', 'sample_duration']
pm25_parts = []
for chunk in pd.read_csv(RAW_PATH, usecols=columns, chunksize=250_000):
    selected = chunk[
        (chunk['parameter_code'] == 88101)
        & (chunk['sample_duration'] == '1 HOUR')
        & chunk['poc'].isin(POCS)
        & chunk['date_local'].isin(smoke_dates)
    ].copy()
    if not selected.empty:
        pm25_parts.append(selected)

pm25 = pd.concat(pm25_parts, ignore_index=True)
pm25['sample_measurement'] = pd.to_numeric(pm25['sample_measurement'], errors='coerce')
pm25['hour_key'] = pm25['date_local'].astype(str) + ' ' + pm25['time_local'].astype(str).str[:5]
hourly_pm25 = pm25.pivot_table(index='hour_key', columns='poc', values='sample_measurement', aggfunc='mean')
hourly_pm25['pm25'] = hourly_pm25.reindex(columns=POCS).mean(axis=1)
hourly_pm25['date'] = hourly_pm25.index.str[:10]

valid_days = set(hourly_pm25.loc[hourly_pm25['pm25'].notna(), 'date'])
missing_days = smoke_dates - valid_days
assert not missing_days, f'Smoke days without valid hourly PM2.5: {sorted(missing_days)}'
print(f'Matched {len(hourly_pm25):,} hourly timestamps across {len(valid_days):,} smoke days')

Matched 936 hourly timestamps across 39 smoke days


## Results

In [18]:
daily_max_pm25 = hourly_pm25.groupby('date')['pm25'].max()
days_to_move = set(daily_max_pm25[daily_max_pm25 < THRESHOLD].index)
move_mask = smoke['_date'].isin(days_to_move)
rows_to_move = smoke.loc[move_mask].drop(columns='_date')
filtered_smoke = smoke.loc[~move_mask].drop(columns='_date')
filtered_no_smoke = pd.concat([no_smoke, rows_to_move], ignore_index=True)

rows_moved = len(rows_to_move)
percent_dropped = 100 * rows_moved / len(smoke)
print(f'Days moved: {len(days_to_move):,} of {len(smoke_dates):,}')
print(f'Rows dropped from smoke: {rows_moved:,} of {len(smoke):,} ({percent_dropped:.2f}%)')
print(f'Filtered smoke rows: {len(filtered_smoke):,}')
print(f'Filtered no-smoke rows: {len(filtered_no_smoke):,}')

Days moved: 5 of 39
Rows dropped from smoke: 6 of 48 (12.50%)
Filtered smoke rows: 42
Filtered no-smoke rows: 92


In [20]:
# Reconcile all rows before writing outputs.
assert len(filtered_smoke) + len(filtered_no_smoke) == len(smoke) + len(no_smoke)
assert list(filtered_smoke.columns) == list(smoke.drop(columns='_date').columns)
assert list(filtered_no_smoke.columns) == list(no_smoke.columns)
assert set(filtered_smoke['dt']).isdisjoint(filtered_no_smoke['dt'])
assert not filtered_smoke['dt'].astype(str).str[:10].isin(days_to_move).any()
assert set(rows_to_move['dt']).issubset(set(filtered_no_smoke['dt']))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
filtered_smoke.to_csv(SMOKE_OUTPUT, index=False)
filtered_no_smoke.to_csv(NO_SMOKE_OUTPUT, index=False)
print(f'Wrote {SMOKE_OUTPUT}')
print(f'Wrote {NO_SMOKE_OUTPUT}')

Wrote /Users/isaacsudweeks/Desktop/Work/Improved_Isopleths/data/utah/data_up_to_2025/new_canister_breakout_2026_06_23/Filtered_Canister_data/PM2.5_Smoke/2025_summed_smoke_all_PM2.5_filtered.csv
Wrote /Users/isaacsudweeks/Desktop/Work/Improved_Isopleths/data/utah/data_up_to_2025/new_canister_breakout_2026_06_23/Filtered_Canister_data/PM2.5_Smoke/2025_summed_no_smoke_all_PM2.5_filtered.csv


## Takeaways

The executed cells above record the exact number of affected days and rows, the percentage removed from the smoke dataset, and output paths. A day moves only when its maximum available hourly PM2.5 mean is below 9 µg/m³. The final assertions reconcile row counts and ensure the moved timestamps occur only in the no-smoke output.